# Исследование игровой индустрии

## Описание проекта

Проект посвящён анализу данных о видеоиграх: платформах, жанрах, годах выпуска, продажах в разных регионах, оценках пользователей и критиков, а также возрастных рейтингах ESRB.

**Цель проекта** — подготовить данные за 2000–2013 годы и выделить основные характеристики игровой индустрии за этот период: популярные платформы, качество данных, структуру оценок и категории игр.

## Данные

В датасете представлены следующие признаки:

- `Name` — название игры;
- `Platform` — игровая платформа;
- `Year of Release` — год выпуска;
- `Genre` — жанр;
- `NA sales`, `EU sales`, `JP sales`, `Other sales` — продажи по регионам, млн копий;
- `Critic Score` — оценка критиков;
- `User Score` — оценка пользователей;
- `Rating` — возрастной рейтинг ESRB.

## План работы

1. Загрузить данные и изучить их структуру.
2. Проверить типы данных, пропуски и дубликаты.
3. Выполнить предобработку: переименовать столбцы, привести типы, обработать пропуски и дубликаты.
4. Отобрать данные за 2000–2013 годы.
5. Разделить игры на категории по оценкам пользователей и критиков.
6. Сформулировать итоговые выводы.

## Используемые инструменты

- Python
- pandas

## Загрузка данных и знакомство с ними

In [ ]:
import pandas as pd #выгружаем датасет
df = pd.read_csv('new_games.csv')

In [ ]:
df.info() #выводим первые строки
df.head(15) #выводим инф-ю о датасете

Полученные данные состоят из 11 столбцов, 16956 строк и соответствуют описанию. В 6 столбцах встречаются пропуски. Типы данных используются верно, кроме столбцей "Year of Release" (данные в нем нужно привести к типу с целыми числами (int64)), "EU sales", "JP sales", "User Score", "Rating" - в них тип данных нужно изменить на вещественный тип (float64).
Столбец "Rating" для лаконичности переименуем в "Rating ESRB"

## Проверка ошибок в данных и их предобработка

In [ ]:
df.columns #выводим названия столбцов

In [ ]:
df.columns = df.columns.str.lower().str.replace(' ', '_') #меняем стиль написания на snake case
df = df.rename(columns={'rating': 'rating_esrb'}) #меняем непонятное названние столбца
print(df.columns)

### Типы данных

Предположительные причины некорретных типов данных:

1. человеческие ошибки
2. ошибки при автоматическом определении типа

In [ ]:
column_convert = ['eu_sales', 'jp_sales', 'user_score'] # переменная для изменений типа 
for column in column_convert: #цикл преобразует строковые значения в NaN
    df[column] = pd.to_numeric(df[column], errors='coerce')
df['year_of_release'] = df['year_of_release'].astype('Int32') #явно приводим столбец к Int для сохранения NaN

In [ ]:
null_sum = df.isna().sum() #считаем абсолютное кол-во пропусков в каждом столбце
display(null_sum)

In [ ]:
relative_value = null_sum / len(df)*100 #считаем относительное кол-во пропусков в каждом столбце
display(relative_value)

Пропуски харктерны для столбцов, в которых хранятся данные о рейтинге игр: 
1. critic_score (возможно, потому что не на все игры есть рецензии критиков)
2. user_score (возможно, потому что не каждый пользователь оценивает игры)
3. rating_esrb (возможно, потому что оцениваемые игры вышли раньше, чем появился рейтинг организации ESRB)

Незначительный (в сравнении предыдущих столбцов) процент пропусков приходится на стоблец с годами выпуска игр. Это возможно, потому что произошла ошибка (техническая либо человеческий фактор). Очень малое кол-во пропусков в столбцах с названнием игры (0.01%), в столбцах с продажами в Японии (0.02%) и Европе (0.04%)

Процент пропусков в столбце с годами выпуска игр невысокий (1.6%), поэтому строки с пропусками мы удалим, чтобы не искажать данные. С этой же целью относительно выбросов, в столбцах с рейтингом заполним пропуски медианным значением. Строки с пропущенными названиями мы тоже удалим, это не повлияет на анализ. Пропуски в столбах с продажами заполним средним значением от года выхода игры.

In [ ]:
df = df.dropna(subset=['year_of_release', 'name']) #удаляем пропуски в столбце с годами и названиями

In [ ]:
df[['critic_score', 'user_score']] = df[['critic_score', 'user_score']].fillna(df[['critic_score', 'user_score']].median()) #заполняем пропуски в столбцах с рейтингом

In [ ]:
for sales_column in ['eu_sales', 'jp_sales']: #цикл перебирает нужные столбцы, в которых есть пропуски, заполняет медианным значнеием в зависимости от года
    avg_by_year = df.groupby('year_of_release')[sales_column].median()
    for year in avg_by_year.index:
        df.loc[(df['year_of_release'] == year) & (df[sales_column].isna()), sales_column] = avg_by_year[year]

In [ ]:
df['rating_esrb'] = df['rating_esrb'].fillna(df['rating_esrb'].mode()[0]) #заполняем пропуски самым часто встречающимся рейтингом

### Явные и неявные дубликаты в данных

Изучаем уникальные значения в категориальных данных

In [ ]:
categorical_columns = ['genre', 'platform', 'rating_esrb', 'year_of_release'] #создаем переменную с нужными столбцами
for column in categorical_columns: #цикл выводит уникальные значения в каждом столбце
    print(f"{column}")
    print(df[column].unique())

In [ ]:
df['genre'] = df['genre'].str.lower() #неявные дубликаты обнаружены в столбце с жанрами. проведем нормализацию данных
                                       #к нижнему регистру

In [ ]:
duplicates_all = df[df.duplicated(keep=False)] #в этой переменной сохраняем все явные дублирующиеся строки
print(f"Всего дублирующихся строк: {len(duplicates_all)}") #находим кол-во дубликатов
display(duplicates_all)

Кол-во найденных явных дубликатов - 470 строк. Явные дубликаты по своей природе являются абсолютно идентичными, поэтому применим методы pandas для их устранения

In [ ]:
df = df.drop_duplicates(keep='first') #удаляем явные дубликаты с сохранением первой строки

В исходном датасете было 16 956 строк. Воспользуемся этим числом

In [ ]:
before_len = 16956
after_len = 16956 - len(df) #вычисляем абсолютное значение удаленных строк путем вычитания нанешнего кол-ва строк из изначального
relative_len = round (after_len / before_len * 100, 2) #вычисляем относительное кол-во удаленных строк, округляем
display(f"Всего строк удалено: {after_len}")
display(f"Кол-во удаленных строк относительно изначального (%): {relative_len}")

На этапе предобработки данных мы:
1. Нормализовали стиль написания столбцов
2. Преобразовали данные в нужный тип
3. Обработали пропуски и дубликаты в данных

Большинство пропусков удалось заполнить. Количество дубликатов оказалось небольшим(470 строк), как и количество удаленных строк (512 или 3.02% от всех строк). Этот процент говорит о том, что данные почти не потеряны и результат не будет искажен.


## Фильтрация данных

In [ ]:
df_actual = df[(df['year_of_release'] >= 2000) & (df['year_of_release'] <= 2013)] #в новом датафрейме сохраняем данные за 2000-2013 года

## Категоризация данных

Разделим игры на категории по оценкам пользователей и критиков. Это поможет быстрее сравнивать игры не только по числовым оценкам, но и по качественным группам.

Категории по оценкам пользователей:

- низкая оценка: от 0 до 3;
- средняя оценка: от 3 до 8;
- высокая оценка: от 8 до 10.

Категории по оценкам критиков:

- низкая оценка: от 0 до 30;
- средняя оценка: от 30 до 80;
- высокая оценка: от 80 до 100.

In [ ]:
df_actual = df_actual.copy() #pandas дал предупреждение SettingWithCopyWarning, создала копию

In [ ]:
df_actual['user_score_category'] = pd.cut( #категоризируем игры по оценкам пользователей
    df_actual['user_score'],
    bins=[0, 3, 8, 10], #расставляем границы
    labels=['низкая оценка', 'средняя оценка', 'высокая оценка'], #именуем категории
    right=False)  # правый край интервала не включается (кроме последнего)

In [ ]:
df_actual['critic_score_category'] = pd.cut( #все аналогично коду выше, только по оценке критиков
    df_actual['critic_score'],
    bins=[0, 30, 80, 100],
    labels=['низкая оценка', 'средняя оценка', 'высокая оценка'],
    right=False)

In [ ]:
display("По оценкам пользователей") #выводим текст
display(df_actual.groupby('user_score_category', observed=False)['user_score'].count()) #группируем по категориям и считаем количество игр в них

In [ ]:
display("По оценкам критиков") #аналогично коду выше, только для оценки критиков
display(df_actual.groupby('critic_score_category', observed=False)['critic_score'].count())

In [ ]:
count_game = df_actual['platform'].value_counts() #считаем кол-во игр по платформам. 
#данный метод сортирует по умолчанию от большего к меньшему

In [ ]:
display(count_game.head(7)) #поэтому просто выводим первые 7 платформ

---

## Итоговый вывод

В ходе выполнения проекта мы провели полную предобработку данных об игровых платформах, подготовили срез за период 2000–2013 годы и добавили новые категориальные признаки на основе оценок.

Описание этапов:
1. Загрузили данные, исправили типы столбцов (продажи, оценки, год выпуска), привели названия к удобному формату
2. Обработали пропуски: удалили строки без года и названия, остальные заполнили медианами и модой
3. Убрали дубликаты — явные (470 шт.), обработали неявные в жанрах, что позволило не исказить данные при анализе
4. Отфильтровали период 2000–2013 годов в отдельный датафрейм `df_actual`, в него же вошли новые столбцы (`user_score_category` и `critic_score_category`) с категориями оценки критиков и пользователей (низкая, средняя, высокая)

За данный временной период самыми популярными по количеству выходов игр платформами оказались:
- PS2 (2127 игр)
- DS (2120)
- Wii (1275)
Подготовленный датасет можно использовать для дальнейшего анализа рынка: сравнения платформ, жанров, региональных продаж, пользовательских и критических оценок.